# CURE-Rec — SASRec external robustness evaluation

Run this notebook **only after** the frozen BPR archive and the CURE-Sim calibration study are accepted. SASRec is an external sequential-ranking comparator. It is not used to make CURE-Sim causal claims.

All expensive cells are disabled by default. The search uses validation NDCG@10 only; the final audit is the first held-out test ranking; seed replication uses a fixed configuration without retuning.


## 1. Setup and version guard


In [ ]:
from pathlib import Path
import importlib
import inspect
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.data import load_dataset
from cure_rec.models import chronological_leave_one_out
from cure_rec.sasrec import torch_available
from cure_rec.sasrec_search import SASRecSearchConfig, run_final_sasrec_audit, run_final_sasrec_seed_replication, run_staged_sasrec_search

if not torch_available():
    raise ImportError("SASRec needs PyTorch. In the CURE-Rec virtual environment run: python -m pip install -e '.[dev,torch]'")

RUN_ROOT = ROOT / 'runs'
MOVIELENS_SOURCE = ROOT / 'data' / 'raw' / 'movielens_1m'
SASREC_SEARCH_OUTPUT = RUN_ROOT / 'sasrec-search-movielens'
SASREC_AUDIT_OUTPUT = RUN_ROOT / 'final-sasrec-audit-movielens'
SASREC_SEED_OUTPUT = RUN_ROOT / 'final-sasrec-seed-replication'
print('CURE-Rec source:', ROOT)


## 2. Protocol lock

The split, candidates, and audit are intentionally inherited from the accepted BPR workflow:

- chronological leave-one-out splitting;
- candidates are warm training-catalog items excluding each user’s training-seen items;
- cold held-out targets are counted rather than silently discarded;
- audit asserts zero candidate leakage, missing-target, and ranking-direction violations;
- no test metric appears in the search-selection table.


In [ ]:
RUN_SASREC_SEARCH = False
RUN_FINAL_SASREC_AUDIT = False
RUN_FINAL_SASREC_SEED_REPLICATION = False
SASREC_SEEDS = (42, 43, 44, 45, 46)

assert sum((RUN_SASREC_SEARCH, RUN_FINAL_SASREC_AUDIT, RUN_FINAL_SASREC_SEED_REPLICATION)) <= 1, 'Run one expensive action at a time.'


## 3. Action 1 — validation-only staged SASRec search

This checkpoints completed Stage A/B configurations in `SASREC_SEARCH_OUTPUT` so an interrupted run can resume. It writes validation results and a frozen selection manifest; it does **not** rank held-out test targets.


In [ ]:
if RUN_SASREC_SEARCH:
    sasrec_data = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    sasrec_split = chronological_leave_one_out(sasrec_data.interactions)
    sasrec_selection = run_staged_sasrec_search(
        sasrec_split,
        SASREC_SEARCH_OUTPUT,
        SASRecSearchConfig(stage_epochs=30, final_epochs=120, max_eval_users=1_000, top_k_stage_a=2, seed=42),
    )
    print('SASRec validation-selection output:', SASREC_SEARCH_OUTPUT)
    display(sasrec_selection)
else:
    print('SASRec search disabled.')


## 4. Action 2 — final frozen-configuration SASRec audit

Enable only after reviewing and accepting the validation-selected manifest. This is the first held-out test evaluation for SASRec; it writes shared-candidate audit, pairwise accuracy, validation checkpoint history, and configuration provenance.


In [ ]:
if RUN_FINAL_SASREC_AUDIT:
    sasrec_data = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    sasrec_split = chronological_leave_one_out(sasrec_data.interactions)
    sasrec_audit = run_final_sasrec_audit(
        sasrec_split, SASREC_SEARCH_OUTPUT, SASREC_AUDIT_OUTPUT, seed=42, max_eval_users=1_000
    )
    print('Final SASRec audit:', SASREC_AUDIT_OUTPUT)
    display(sasrec_audit)
else:
    print('Final SASRec audit disabled.')


## 5. Action 3 — fixed-configuration SASRec seed replication

Enable only after the final audit gate passes. The selected architecture and optimizer are frozen; this action re-trains only across independent seeds and produces paired differences against popularity.


In [ ]:
if RUN_FINAL_SASREC_SEED_REPLICATION:
    sasrec_data = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    sasrec_split = chronological_leave_one_out(sasrec_data.interactions)
    sasrec_seeds = run_final_sasrec_seed_replication(
        sasrec_split, SASREC_SEARCH_OUTPUT, SASREC_SEED_OUTPUT, seeds=SASREC_SEEDS, max_eval_users=1_000
    )
    if sasrec_seeds.empty:
        raise RuntimeError('SASRec seed replication returned no paired rows.')
    print('Final SASRec seed replication:', SASREC_SEED_OUTPUT)
    display(sasrec_seeds)
else:
    print('SASRec seed replication disabled.')


## 6. Action 4 — inspect completed assets without training


In [ ]:
if SASREC_SEARCH_OUTPUT.exists():
    search_manifest = json.loads((SASREC_SEARCH_OUTPUT / 'sasrec_search_manifest.json').read_text())
    print(json.dumps(search_manifest, indent=2))
else:
    print('No SASRec search manifest yet.')

if (SASREC_SEED_OUTPUT / 'final_sasrec_seed_summary.csv').exists():
    display(pd.read_csv(SASREC_SEED_OUTPUT / 'final_sasrec_seed_summary.csv'))
else:
    print('No completed SASRec seed summary yet.')


## Interpretation gate

Compare SASRec, BPR, and popularity only under the common evaluator. State that MovieLens supports descriptive chronological-ranking robustness, not causal intervention effects. Archive the SASRec audit and fixed-configuration seed artifacts only after all audit violations are zero.
